# 03 — Entrenamiento y evaluación de YOLO | AI Cook

**Objetivo:** entrenar un detector de ingredientes a partir de `ai_cook_clean` y evaluar su desempeño sobre un conjunto de test independiente.

- Dataset: **360 train / 103 valid / 51 test**, **30 clases**.
- Modelo inicial: **YOLO11n** con pesos preentrenados.
- Entrenamiento: máximo **50 épocas**, imágenes de **640 px**, batch **16**.
- Los parámetros de augmentación se mantienen en sus valores por defecto (sin experimentos adicionales).
- El conjunto de **test no se utiliza para entrenar ni elegir el modelo**.

> En Colab: **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU**.


## 1. Instalación y configuración del entorno

In [ ]:
%pip -q install ultralytics pyyaml pandas matplotlib

In [ ]:
import os
import json
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import yaml
from ultralytics import YOLO
import ultralytics

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("PyTorch:", torch.__version__)
print("Ultralytics:", ultralytics.__version__)
print("GPU disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("Activá GPU en Colab antes de entrenar.")

## 2. Acceso a Drive y ubicación del dataset

**Cambiar `DATASET_SOURCE`** por la carpeta real donde guardaron `ai_cook_clean`. Debe contener `train`, `valid` (o `val`) y `test`, con subcarpetas `images` y `labels`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# MODIFICAR ESTA RUTA si la carpeta está en otro lugar de Drive.
DATASET_SOURCE = Path("/content/drive/MyDrive/ai_cook_clean")

# Ejemplo alternativo si está dentro del repositorio:
# DATASET_SOURCE = Path("/content/drive/MyDrive/vision/artificial/datasets/ai_cook_clean")

assert DATASET_SOURCE.is_dir(), f"No se encontró el dataset en {DATASET_SOURCE}"
print("Dataset encontrado:", DATASET_SOURCE)

### Copiar dataset al disco de Colab

Copiarlo a `/content` suele hacer el entrenamiento más rápido que leer todas las imágenes desde Drive. Los resultados finales se copiarán a Drive.

In [ ]:
DATASET_DIR = Path("/content/ai_cook_clean")
if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
shutil.copytree(DATASET_SOURCE, DATASET_DIR)

VALID_FOLDER = "valid" if (DATASET_DIR / "valid").exists() else "val"
for split in ["train", VALID_FOLDER, "test"]:
    for subdir in ["images", "labels"]:
        p = DATASET_DIR / split / subdir
        assert p.is_dir(), f"Falta la carpeta {p}"

for split in ["train", VALID_FOLDER, "test"]:
    images = [p for p in (DATASET_DIR / split / "images").iterdir()
              if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}]
    print(f"{split}: {len(images)} imágenes")

print("Carpetas verificadas.")

## 3. Preparación del `data.yaml`

Se respetan los nombres de las 30 clases originales, tal como aparecen en el archivo YAML exportado. No redefinir las clases a mano, porque deben coincidir con los IDs de las etiquetas.

In [ ]:
yaml_candidates = [DATASET_DIR / "data.yaml", DATASET_DIR / "dataset.yaml"]
original_yaml = next((p for p in yaml_candidates if p.is_file()), None)
assert original_yaml is not None, "No aparece data.yaml ni dataset.yaml en ai_cook_clean"

with original_yaml.open("r", encoding="utf-8") as f:
    original_config = yaml.safe_load(f)

names = original_config["names"]
if isinstance(names, dict):
    names = [names[k] for k in sorted(names, key=lambda x: int(x))]

assert len(names) == 30, f"Se esperaban 30 clases, pero el YAML tiene {len(names)}"

config = {
    "path": str(DATASET_DIR),
    "train": "train/images",
    "val": f"{VALID_FOLDER}/images",
    "test": "test/images",
    "nc": len(names),
    "names": names,
}

DATA_YAML = Path("/content/ai_cook_yolo.yaml")
with DATA_YAML.open("w", encoding="utf-8") as f:
    yaml.safe_dump(config, f, sort_keys=False, allow_unicode=True)

print(DATA_YAML.read_text(encoding="utf-8"))

## 4. Parámetros y entrenamiento

In [ ]:
MODEL_NAME = "yolo11n.pt"  # baseline preentrenado
EPOCHS = 50
IMAGE_SIZE = 640
BATCH_SIZE = 16
PATIENCE = 15
DEVICE = 0

RUN_PROJECT = Path("/content/yolo_results")
RUN_NAME = "yolo11n_ai_cook_clean"

model = YOLO(MODEL_NAME)
train_results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    patience=PATIENCE,
    device=DEVICE,
    seed=SEED,
    deterministic=True,
    pretrained=True,
    project=str(RUN_PROJECT),
    name=RUN_NAME,
    exist_ok=True,
    plots=True,
    save=True,
    verbose=True,
)

RUN_DIR = RUN_PROJECT / RUN_NAME
BEST_WEIGHTS = RUN_DIR / "weights" / "best.pt"
assert BEST_WEIGHTS.exists(), f"No se encontró {BEST_WEIGHTS}"
print("Mejores pesos:", BEST_WEIGHTS)

## 5. Curvas de entrenamiento

`results.csv` guarda las pérdidas y métricas por época. Los gráficos automáticos de Ultralytics también quedan en la carpeta del experimento.

In [ ]:
csv_path = RUN_DIR / "results.csv"
assert csv_path.exists(), f"No se encontró {csv_path}"
history = pd.read_csv(csv_path)
history.columns = history.columns.str.strip()
display(history.tail())

metric_groups = [
    ["train/box_loss", "val/box_loss"],
    ["train/cls_loss", "val/cls_loss"],
    ["metrics/precision(B)", "metrics/recall(B)"],
    ["metrics/mAP50(B)", "metrics/mAP50-95(B)"],
]

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, columns in zip(axes.ravel(), metric_groups):
    for col in columns:
        if col in history.columns:
            ax.plot(history["epoch"], history[col], label=col)
    ax.set_xlabel("Época")
    ax.set_title(" / ".join(columns))
    ax.legend()
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 6. Evaluación final en test

Se carga `best.pt` y se evalúa una sola vez sobre las 51 imágenes de test. Las métricas se calculan con las anotaciones originales.

In [ ]:
best_model = YOLO(str(BEST_WEIGHTS))

test_metrics = best_model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    plots=True,
    project=str(RUN_PROJECT),
    name="test_evaluation",
    exist_ok=True,
)

summary = {
    "precision": float(test_metrics.box.mp),
    "recall": float(test_metrics.box.mr),
    "mAP50": float(test_metrics.box.map50),
    "mAP50_95": float(test_metrics.box.map),
    "n_train": 360,
    "n_val": 103,
    "n_test": 51,
    "num_classes": len(names),
    "model": MODEL_NAME,
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "seed": SEED,
}

print(pd.DataFrame([summary]).T.to_string(header=False))

with (RUN_DIR / "test_metrics.json").open("w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

## 7. Resultados por ingrediente

In [ ]:
# maps devuelve AP@[0.50:0.95] por clase, en el orden de los IDs.
class_ap = test_metrics.box.maps
per_class = pd.DataFrame({
    "class_id": list(range(len(names))),
    "ingredient": names,
    "mAP50_95": class_ap,
}).sort_values("mAP50_95", ascending=False)

display(per_class)
per_class.to_csv(RUN_DIR / "test_per_class.csv", index=False)

plt.figure(figsize=(10, 9))
plt.barh(per_class["ingredient"], per_class["mAP50_95"])
plt.gca().invert_yaxis()
plt.xlabel("mAP@0.50:0.95")
plt.title("Desempeño por ingrediente en test")
plt.tight_layout()
plt.show()

## 8. Matriz de confusión y predicciones visuales

In [ ]:
from IPython.display import display, Image as DisplayImage

eval_dir = RUN_PROJECT / "test_evaluation"
for filename in ["confusion_matrix.png", "confusion_matrix_normalized.png", "PR_curve.png"]:
    image_path = eval_dir / filename
    if image_path.exists():
        print(filename)
        display(DisplayImage(filename=str(image_path), width=800))

In [ ]:
test_images_dir = DATASET_DIR / "test" / "images"

prediction_results = best_model.predict(
    source=str(test_images_dir),
    imgsz=IMAGE_SIZE,
    conf=0.25,
    device=DEVICE,
    save=True,
    project=str(RUN_PROJECT),
    name="test_predictions",
    exist_ok=True,
    verbose=False,
)

prediction_dir = RUN_PROJECT / "test_predictions"
example_paths = sorted([
    p for p in prediction_dir.iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
])[:6]

for path in example_paths:
    display(DisplayImage(filename=str(path), width=550))

## 9. Guardar resultados y modelo en Drive

Se guardan pesos, métricas, gráficos y predicciones para el informe. **No subir toda la carpeta de datos al repositorio.**

In [ ]:
DRIVE_OUTPUT = Path("/content/drive/MyDrive/vision_artificial_yolo")
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)

# Copiamos las salidas de train, test y predict.
for folder in [RUN_DIR, RUN_PROJECT / "test_evaluation", RUN_PROJECT / "test_predictions"]:
    if folder.exists():
        target = DRIVE_OUTPUT / folder.name
        if target.exists():
            shutil.rmtree(target)
        shutil.copytree(folder, target)

shutil.copy2(DATA_YAML, DRIVE_OUTPUT / "ai_cook_yolo.yaml")

print("Resultados guardados en:", DRIVE_OUTPUT)
print("Modelo entrenado:", DRIVE_OUTPUT / RUN_NAME / "weights" / "best.pt")

## 10. Registro para reproducibilidad

Para comparar con RT-DETR es importante documentar: versión de librerías, GPU, tamaño de imagen, particiones, semilla, batch, número de parámetros, latencia y métricas sobre **el mismo test**. El tiempo/FPS se debe medir después, bajo condiciones equivalentes y con un protocolo común para ambos modelos.


In [ ]:
params = sum(p.numel() for p in best_model.model.parameters())
run_info = {
    **summary,
    "parameters": int(params),
    "epochs_completed": int(len(history)),
    "torch_version": torch.__version__,
    "ultralytics_version": ultralytics.__version__,
    "gpu": torch.cuda.get_device_name(0),
}

with (DRIVE_OUTPUT / "experiment_info.json").open("w", encoding="utf-8") as f:
    json.dump(run_info, f, ensure_ascii=False, indent=2)

print(json.dumps(run_info, ensure_ascii=False, indent=2))